# Demo: Unterrichtsblock 5

Kurze Transferdemo für Imputationen, Joins und Pivot/Melt.

Du arbeitest hier mit Messdaten aus einem kleinen Gasnetz, in das Wasserstoff beigemischt wird: eine Einspeisung und drei Messstellen A, B und C messen den H2-Gehalt in Vol.-%. Anschliessend bearbeitest du selbstständig `01-imputation-joins.ipynb`.

Relevante Artikel in der Wissensdatenbank:

- [Wissensdatenbank: Imputation](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/imputation/)
- [Wissensdatenbank: Joins](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/joins/)
- [Wissensdatenbank: Pivot und Melt](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/pivot_melt/)

In [3]:
import pandas as pd

### Aufgabe 1
Bereite die Rohdaten von Messstelle A in `ub5_demo_h2_messstelle_a_roh.csv` auf und imputiere fehlende Werte nur dort, wo es fachlich vertretbar ist.

- Zeitstempel umwandeln, sortieren
- `status` ungültig: Kalibrierung, Wert unbrauchbar
- einzelne fehlende Werte bei Status ok
- 13:00 bis 16:00 am 11.03.: Kommunikationsausfall, keine Zeilen
- interpolieren erlaubt? wo nicht?

In [14]:
df = pd.read_csv("../data/ub5_demo_h2_messstelle_a_roh.csv")

df

,timestamp,h2_prozent,status
0,2025-03-10 04:45:00,0.51,ok
1,2025-03-10 10:30:00,9.96,ok
2,2025-03-11 17:15:00,9.99,ok
3,2025-03-10 19:30:00,10.00,ok
4,2025-03-11 12:15:00,10.22,ok
...,...,...,...
175,2025-03-10 17:45:00,NaN,ok
176,2025-03-11 02:30:00,9.82,ok
177,2025-03-10 03:30:00,0.38,ok
178,2025-03-10 23:00:00,10.01,ok


In [22]:
# Dein Code hier

df["timestamp"] = pd.to_datetime(df["timestamp"])
df.set_index("timestamp").sort_index()

df.describe()

,timestamp,h2_prozent
count,180,180.000000
mean,2025-03-10 22:54:30,7.621444
min,2025-03-10 00:00:00,0.000000
25%,2025-03-10 11:11:15,5.166250
50%,2025-03-10 22:22:30,9.950000
75%,2025-03-11 09:33:45,10.030000
max,2025-03-11 23:45:00,10.250000
std,NaN,4.116052


In [16]:
(df["status"]!="ok").sum()

np.int64(3)

In [17]:
df[df["status"]!="ok"]

,timestamp,h2_prozent,status
17,2025-03-11 07:15:00,0.0,ungültig
33,2025-03-11 07:30:00,0.0,ungültig
42,2025-03-11 07:00:00,0.0,ungültig


In [18]:
df[df["status"]=="ok"].copy()

,timestamp,h2_prozent,status
0,2025-03-10 04:45:00,0.51,ok
1,2025-03-10 10:30:00,9.96,ok
2,2025-03-11 17:15:00,9.99,ok
3,2025-03-10 19:30:00,10.00,ok
4,2025-03-11 12:15:00,10.22,ok
...,...,...,...
175,2025-03-10 17:45:00,NaN,ok
176,2025-03-11 02:30:00,9.82,ok
177,2025-03-10 03:30:00,0.38,ok
178,2025-03-10 23:00:00,10.01,ok


In [21]:
df.isna().sum()

timestamp     0
h2_prozent    0
status        0
dtype: int64

In [20]:
df["h2_prozent"]=df["h2_prozent"].interpolate(method="linear")

### Aufgabe 2
Führe `ub5_demo_h2_einspeisung.csv` und `ub5_demo_h2_messstelle_a.csv` über die Zeit zusammen, damit beide H2-Gehalte nebeneinander stehen.

- Join-Arten vergleichen: `inner`, `left`, `right`, `outer`
- Einspeisung alle 15 min, Messstelle A nur stündlich
- welche Zeitpunkte bleiben jeweils erhalten?
- welche Join-Art behält alle Zeitpunkte der Einspeisung?

In [26]:
einspeisung = pd.read_csv("../data/ub5_demo_h2_einspeisung.csv")
messstelle_a = pd.read_csv("../data/ub5_demo_h2_messstelle_a.csv")

print("Einspeisung")
display(einspeisung.head(3))
print("Messstelle A")
display(messstelle_a.head(3))

Einspeisung


,timestamp,h2_einspeisung
0,2025-03-10 00:00:00,0.20
1,2025-03-10 00:15:00,0.14
2,2025-03-10 00:30:00,0.27


Messstelle A


,timestamp,h2_messstelle_a
0,2025-03-10 00:00:00,0.20
1,2025-03-10 01:00:00,0.30
2,2025-03-10 02:00:00,0.24


In [27]:
# Dein Code hier
messstelle_a= messstelle_a.set_index("timestamp")
einspeisung=einspeisung.set_index("timestamp")
einspeisung.join(messstelle_a, how="right")

,h2_einspeisung,h2_messstelle_a
timestamp,,
2025-03-10 00:00:00,0.20,0.20
2025-03-10 01:00:00,0.44,0.30
2025-03-10 02:00:00,0.39,0.24
2025-03-10 03:00:00,0.19,0.44
2025-03-10 04:00:00,0.29,0.22
2025-03-10 05:00:00,0.21,0.40
2025-03-10 06:00:00,0.35,0.29
2025-03-10 07:00:00,0.18,0.63
2025-03-10 08:00:00,10.00,0.28


### Aufgabe 3
Wandle den breiten Datensatz `ub5_demo_h2_netz_wide.csv` mit `melt` in ein langes Format um, mit einer Spalte für die Messstelle.

In [4]:
df_wide = pd.read_csv("../data/ub5_demo_h2_netz_wide.csv")

df_wide.head(3)

,timestamp,einspeisung,messstelle_a,messstelle_b,messstelle_c
0,2025-03-10 00:00:00,0.20,0.20,1.20,0.13
1,2025-03-10 01:00:00,0.44,0.30,1.44,0.02
2,2025-03-10 02:00:00,0.39,0.24,1.18,0.36


In [5]:
df_long = pd.melt(df_wide, id_vars="timestamp", var_name="messort", value_name="value")
df_long

,timestamp,messort,value
0,2025-03-10 00:00:00,einspeisung,0.20
1,2025-03-10 01:00:00,einspeisung,0.44
2,2025-03-10 02:00:00,einspeisung,0.39
3,2025-03-10 03:00:00,einspeisung,0.19
4,2025-03-10 04:00:00,einspeisung,0.29
...,...,...,...
187,2025-03-11 19:00:00,messstelle_c,10.08
188,2025-03-11 20:00:00,messstelle_c,9.95
189,2025-03-11 21:00:00,messstelle_c,9.92
190,2025-03-11 22:00:00,messstelle_c,10.10


### Aufgabe 4
Wandle den langen Datensatz `ub5_demo_h2_netz_long.csv` mit `pivot` zurück in ein breites Format um.

In [7]:
df_long = pd.read_csv("../data/ub5_demo_h2_netz_long.csv")

df_long.head(8)

,timestamp,messstelle,h2_prozent
0,2025-03-10 00:00:00,einspeisung,0.20
1,2025-03-10 01:00:00,einspeisung,0.44
2,2025-03-10 02:00:00,einspeisung,0.39
3,2025-03-10 03:00:00,einspeisung,0.19
4,2025-03-10 04:00:00,einspeisung,0.29
5,2025-03-10 05:00:00,einspeisung,0.21
6,2025-03-10 06:00:00,einspeisung,0.35
7,2025-03-10 07:00:00,einspeisung,0.18


In [8]:
# Dein Code hier
df_wide2 = df_long.pivot(index="timestamp", columns="messstelle", values="h2_prozent")
df_wide2

messstelle,einspeisung,messstelle_a,messstelle_b,messstelle_c
timestamp,,,,
2025-03-10 00:00:00,0.20,0.20,1.20,0.13
2025-03-10 01:00:00,0.44,0.30,1.44,0.02
2025-03-10 02:00:00,0.39,0.24,1.18,0.36
2025-03-10 03:00:00,0.19,0.44,1.29,0.40
2025-03-10 04:00:00,0.29,0.22,1.49,0.29
2025-03-10 05:00:00,0.21,0.40,1.42,0.15
2025-03-10 06:00:00,0.35,0.29,1.27,0.54
2025-03-10 07:00:00,0.18,0.63,1.33,0.22
2025-03-10 08:00:00,10.00,0.28,1.47,0.20
